In [ ]:
import pandas as pd 
import numpy as np
from matplotlib import pyplot as plt

from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm import tqdm

## Batch 1 Preprocessing

### Luminiscense Data

In [ ]:
batch_1_lumi_data = pd.read_excel("data/tb_data/raw_data/Batch1_lumi.xlsx")
batch_1_lumi_data

### Resazurin Data

In [ ]:
batch_1_resazurin_data = pd.read_excel("data/tb_data/raw_data/Batch1_resazurin.xlsx")
print("Raw Batch 1 data contains: ", len(batch_1_resazurin_data), 'molecules')
batch_1_resazurin_data = batch_1_resazurin_data[['SMILES','Mtb_SAC3HTS_20uM_7H9DextroseAcetate_Resazurin: Raw Data % inhibition (%)']]
batch_1_resazurin_data = batch_1_resazurin_data.rename(columns={'Mtb_SAC3HTS_20uM_7H9DextroseAcetate_Resazurin: Raw Data % inhibition (%)':'inhibition'})
batch_1_resazurin_data = batch_1_resazurin_data.dropna()
print("Raw Batch 1 data contains: ", len(batch_1_resazurin_data), "molecules after dropping molecules without SMILES structure")
batch_1_resazurin_data


In [ ]:
batch_1_resazurin_data[batch_1_resazurin_data.duplicated(subset = ['SMILES'], keep = False)] ## No duplicating molecules in batch 1 

In [ ]:
plt.hist(batch_1_resazurin_data['inhibition'].sort_values(), bins = 100)
plt.show()

## Batch 2 Preprocessing (Resazurin Only)

In [ ]:
batch_2_resazurin_data = pd.read_excel('data/tb_data/raw_data/Batch2_resazurin.xlsx')
print("Raw Batch 2 data contains: ", len(batch_2_resazurin_data), "molecules")
batch_2_resazurin_data = batch_2_resazurin_data[['SMILES',"Sac2 HTS MC27000 on M9+NaOAc+Dextrose: Raw Data % inhibition (%)"]]
batch_2_resazurin_data = batch_2_resazurin_data.rename(columns={'Sac2 HTS MC27000 on M9+NaOAc+Dextrose: Raw Data % inhibition (%)':'inhibition'})
batch_2_resazurin_data = batch_2_resazurin_data.dropna()
print("Raw Batch 2 data contains: ", len(batch_2_resazurin_data), "molecules after dropping molecules without SMILES structure")

batch_2_duplicated = batch_2_resazurin_data[batch_2_resazurin_data.duplicated(subset = ['SMILES'], keep = False)]

In [ ]:
plt.hist(batch_2_duplicated.groupby('SMILES').diff().dropna()['inhibition'].abs())
plt.show()

In [ ]:
confliction_threshold = 20

conflict_index = batch_2_duplicated.groupby(['SMILES'], as_index = True).diff().dropna()['inhibition'].abs()
conflict_index = conflict_index[conflict_index > confliction_threshold].index
conflict_smiles = set(batch_2_resazurin_data.loc[conflict_index]['SMILES'])

print(len(conflict_smiles))

In [ ]:
batch_2_resazurin_data = batch_2_resazurin_data[~batch_2_resazurin_data['SMILES'].isin(conflict_smiles)]
print("Raw Batch 2 data contains: ", len(batch_2_resazurin_data), "molecules after dropping conflicting inhibition readouts")

batch_2_resazurin_data = batch_2_resazurin_data.groupby(['SMILES'], as_index = False).mean()
print("Raw Batch 2 data contains: ", len(batch_2_resazurin_data), "molecules after averaging duplicated readouts")

In [ ]:
plt.hist(batch_2_resazurin_data['inhibition'].sort_values(), bins=100)
plt.show()

## Concatenating Batch 1 & 2

In [ ]:
batch1_and_2_resazurin_data = pd.concat([batch_1_resazurin_data,batch_2_resazurin_data])
batch1_and_2_resazurin_data

In [ ]:
batch1_and_2_resazurin_data[batch1_and_2_resazurin_data.duplicated(subset=['SMILES'])] ## No duplicated molecules between set 1 and 2

In [ ]:
plt.hist(batch1_and_2_resazurin_data['inhibition'].sort_values(), bins=100)
plt.show()

In [ ]:
## Min-Max Normalization
batch1_and_2_resazurin_data['normalized_inhibition'] = (batch1_and_2_resazurin_data['inhibition'] - batch1_and_2_resazurin_data['inhibition'].min()) / (batch1_and_2_resazurin_data['inhibition'].max() - batch1_and_2_resazurin_data['inhibition'].min())
batch1_and_2_resazurin_data

In [ ]:
plt.hist(batch1_and_2_resazurin_data['normalized_inhibition'].sort_values(), bins=100)
plt.show()

### Generate 3d Structure

In [ ]:
smiles = [smi for smi in batch1_and_2_resazurin_data.SMILES]
mols = [Chem.MolFromSmiles(smi) for smi in batch1_and_2_resazurin_data.SMILES]
tb_inhibition = [label for label in batch1_and_2_resazurin_data.inhibition]
normalized_tb_inhibition = [label for label in batch1_and_2_resazurin_data.normalized_inhibition]

hmols = []
for m in range(len(mols)):
    if mols[m] is not None:
        hmols.append(Chem.AddHs(mols[m]))
    else:
        smiles[m] = None
        tb_inhibition[m] = None
        normalized_tb_inhibition[m] = None
        continue

smiles = [i for i in smiles if i is not None]
tb_inhibition = [i for i in tb_inhibition if i is not None]
normalized_tb_inhibition = [i for i in normalized_tb_inhibition if i is not None]

print(len(hmols))
print(len(smiles))
print(len(tb_inhibition))
print(len(normalized_tb_inhibition))

print("Resazurin Data contains: ", len(hmols), "molecules after remove the molecules fail to read by rdkit" )

In [ ]:
optimize_status_list = []

for i in tqdm(range(len(hmols))):
    embed_status = AllChem.EmbedMolecule(hmols[i], AllChem.ETKDG())
    if embed_status != 0:
        hmols[i] = None
        smiles[i] = None
        tb_inhibition[i] = None
        normalized_tb_inhibition[i] = None
        continue
    else:
        optimize_status = AllChem.UFFOptimizeMolecule(hmols[i],1000)
        optimize_status_list.append(optimize_status)

        if optimize_status != 0:
            print(f"Optimization Failed on Molecule Index: {i}")
            hmols[i] = None
            smiles[i] = None
            tb_inhibition[i] = None
            normalized_tb_inhibition[i] = None
    
hmols = [i for i in hmols if i is not None]
smiles = [i for i in smiles if i is not None]
tb_inhibition = [i for i in tb_inhibition if i is not None]
normalized_tb_inhibition = [i for i in normalized_tb_inhibition if i is not None]

print(len(hmols))
print(len(smiles))
print(len(tb_inhibition))
print(len(normalized_tb_inhibition))
    
print(f"RDKit Optimization failed on {sum(optimize_status_list)} Molecules")

In [ ]:
batch1_and_2_resazurin_data = batch1_and_2_resazurin_data[batch1_and_2_resazurin_data['SMILES'].isin(smiles)]
batch1_and_2_resazurin_data

In [ ]:
writer = Chem.SDWriter('data/tb_data/raw_data/regression_data/TB_inhibition_resazurin_62498_molecules_3d.sdf')

print(f">>>>> Successfully Converted {len(hmols)} molecules into 3-D Structure >>>>>")

print(">>>>> Saving to File >>>>>")
for n in tqdm(range(len(hmols))):
    hmols[n].SetProp("_SMILES","%s"%smiles[n])
    hmols[n].SetProp("_tb_inhibition","%s"%tb_inhibition[n])
    writer.write(hmols[n])
    
writer.close()

In [ ]:
writer = Chem.SDWriter('data/tb_data/raw_data/regression_data/normalized_TB_inhibition_resazurin_62498_molecules_3d.sdf')

print(f">>>>> Successfully Converted {len(hmols)} molecules into 3-D Structure >>>>>")

print(">>>>> Saving to File >>>>>")
for n in tqdm(range(len(hmols))):
    hmols[n].SetProp("_SMILES","%s"%smiles[n])
    hmols[n].SetProp("_tb_inhibition","%s"%tb_inhibition[n])
    hmols[n].SetProp("_normalized_tb_inhibition", "%s"%normalized_tb_inhibition[n])
    writer.write(hmols[n])
    
writer.close()

In [ ]:
normalized_data = batch1_and_2_resazurin_data[['SMILES','normalized_inhibition']]
standard_data = batch1_and_2_resazurin_data[['SMILES','inhibition']]

standard_data.to_csv("data/tb_data/raw_data/regression_data/TB_inhibition_resazurin_62498_molecules.csv",
                     index = False)
normalized_data.to_csv("data/tb_data/raw_data/regression_data/normalized_TB_inhibition_resazurin_62498_molecules.csv",
                                   index = False)

In [ ]:
standard_data

In [ ]:
normalized_data

In [ ]:
supply = Chem.SDMolSupplier('data/tb_data/raw_data/regression_data/TB_inhibition_resazurin_62498_molecules_3d.sdf')

pos_list = []
atom_feature = []
labels = []
smiles = []
fingerprints = []
binary_labels = []

print(">>>>> Loading Data Files >>>>>")

for idx, mol in enumerate(tqdm(supply, total = len(supply))):
    smiles.append(mol.GetProp('_SMILES'))
    labels.append(float(mol.GetProp('_tb_inhibition')))
    binary_labels.append(1 if float(mol.GetProp('_tb_inhibition')) >= 40 else 0)
    mol_pos_list = []
    mol_atom_feature = []
    for i, atom in enumerate(mol.GetAtoms()):
        positions = mol.GetConformer().GetAtomPosition(i)
        mol_atom_feature.append([atom.GetAtomicNum(), atom.GetMass(), atom.GetDegree(), 
                                 atom.GetTotalDegree(), atom.GetTotalValence(), atom.GetNumRadicalElectrons(),
                                 atom.GetFormalCharge(), 1 if atom.GetIsAromatic() else 0])
        mol_pos_list.append([positions.x, positions.y, positions.z])
    pos_list.append(mol_pos_list)
    atom_feature.append(mol_atom_feature)

In [ ]:
print(smiles[62497])
print(labels[62497])

In [ ]:
print(len(pos_list[62497]))